In [1]:
# ==========================================
# SEKCJA 1
# Zwykła sieć neuronowa na prostych danych
# ==========================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

# dla powtarzalności wyników
np.random.seed(42)
tf.random.set_seed(42)

# -------------------------------------------------
# 1. Tworzymy mały zbiór danych
# Każdy obiekt ma 4 cechy:
# novelty, feasibility, cost_efficiency, stability
# y = 1 oznacza "dobry projekt"
# y = 0 oznacza "słaby projekt"
# -------------------------------------------------

X = np.array([
    [0.90, 0.85, 0.80, 0.75],
    [0.88, 0.80, 0.78, 0.70],
    [0.20, 0.30, 0.40, 0.35],
    [0.25, 0.20, 0.30, 0.25],
    [0.75, 0.70, 0.65, 0.60],
    [0.35, 0.40, 0.45, 0.30],
    [0.92, 0.90, 0.85, 0.88],
    [0.15, 0.25, 0.20, 0.22],
    [0.78, 0.82, 0.74, 0.76],
    [0.28, 0.35, 0.38, 0.32],
    [0.83, 0.79, 0.81, 0.72],
    [0.18, 0.22, 0.33, 0.27],
], dtype=np.float32)

y = np.array([
    1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0
], dtype=np.float32)

feature_names = ["novelty", "feasibility", "cost_efficiency", "stability"]

df = pd.DataFrame(X, columns=feature_names)
df["label"] = y.astype(int)

print("Dane wejściowe:")
print(df)

# -------------------------------------------------
# 2. Budujemy prostą sieć neuronową
# -------------------------------------------------

model_base = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model_base.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# -------------------------------------------------
# 3. Uczymy model
# -------------------------------------------------

history_base = model_base.fit(
    X, y,
    epochs=100,
    verbose=0
)

# -------------------------------------------------
# 4. Predykcje
# -------------------------------------------------

base_pred = model_base.predict(X, verbose=0).flatten()
base_class = (base_pred > 0.5).astype(int)

results_base = df.copy()
results_base["nn_score"] = base_pred
results_base["nn_class"] = base_class

print("\nWyniki zwykłej sieci:")
print(results_base)

Dane wejściowe:
    novelty  feasibility  cost_efficiency  stability  label
0      0.90         0.85             0.80       0.75      1
1      0.88         0.80             0.78       0.70      1
2      0.20         0.30             0.40       0.35      0
3      0.25         0.20             0.30       0.25      0
4      0.75         0.70             0.65       0.60      1
5      0.35         0.40             0.45       0.30      0
6      0.92         0.90             0.85       0.88      1
7      0.15         0.25             0.20       0.22      0
8      0.78         0.82             0.74       0.76      1
9      0.28         0.35             0.38       0.32      0
10     0.83         0.79             0.81       0.72      1
11     0.18         0.22             0.33       0.27      0

Wyniki zwykłej sieci:
    novelty  feasibility  cost_efficiency  stability  label  nn_score  \
0      0.90         0.85             0.80       0.75      1  0.724489   
1      0.88         0.80           

In [2]:
# ==========================================
# SEKCJA 2
# Jeden atraktor jako wzorzec idealny
# ==========================================

# -------------------------------------------------
# 1. Definiujemy jeden atraktor
# To jest ręcznie ustalony punkt idealny
# -------------------------------------------------

attractor_1 = np.array([0.90, 0.85, 0.80, 0.80], dtype=np.float32)

print("Atraktor 1:")
print(attractor_1)

# -------------------------------------------------
# 2. Liczymy odległość euklidesową od atraktora
# -------------------------------------------------

distance_1 = np.linalg.norm(X - attractor_1, axis=1)

# -------------------------------------------------
# 3. Zamieniamy odległość na rezonans
# Im bliżej, tym większy rezonans
# -------------------------------------------------

resonance_1 = 1 / (1 + distance_1)

results_attr1 = df.copy()
results_attr1["distance_to_A1"] = distance_1
results_attr1["resonance_A1"] = resonance_1

print("\nWyniki względem jednego atraktora:")
print(results_attr1.sort_values("resonance_A1", ascending=False))

Atraktor 1:
[0.9  0.85 0.8  0.8 ]

Wyniki względem jednego atraktora:
    novelty  feasibility  cost_efficiency  stability  label  distance_to_A1  \
0      0.90         0.85             0.80       0.75      1        0.050000   
6      0.92         0.90             0.85       0.88      1        0.108628   
1      0.88         0.80             0.78       0.70      1        0.115326   
10     0.83         0.79             0.81       0.72      1        0.122474   
8      0.78         0.82             0.74       0.76      1        0.143178   
4      0.75         0.70             0.65       0.60      1        0.327872   
5      0.35         0.40             0.45       0.30      0        0.936750   
9      0.28         0.35             0.38       0.32      0        1.020392   
2      0.20         0.30             0.40       0.35      0        1.074709   
3      0.25         0.20             0.30       0.25      0        1.182159   
11     0.18         0.22             0.33       0.27      0  

In [3]:
# ==========================================
# SEKCJA 3
# Dwa atraktory
# ==========================================

# -------------------------------------------------
# Atraktor 1: ambitny / mocny / wysoko jakościowy
# Atraktor 2: umiarkowany / stabilny / bardziej zachowawczy
# -------------------------------------------------

attractor_1 = np.array([0.90, 0.85, 0.80, 0.80], dtype=np.float32)
attractor_2 = np.array([0.72, 0.68, 0.62, 0.65], dtype=np.float32)

# odległości
distance_A1 = np.linalg.norm(X - attractor_1, axis=1)
distance_A2 = np.linalg.norm(X - attractor_2, axis=1)

# rezonanse
resonance_A1 = 1 / (1 + distance_A1)
resonance_A2 = 1 / (1 + distance_A2)

# minimalna odległość do najbliższego atraktora
best_distance = np.minimum(distance_A1, distance_A2)

# maksymalny rezonans z dwóch atraktorów
best_resonance = np.maximum(resonance_A1, resonance_A2)

results_attr2 = df.copy()
results_attr2["distance_A1"] = distance_A1
results_attr2["distance_A2"] = distance_A2
results_attr2["resonance_A1"] = resonance_A1
results_attr2["resonance_A2"] = resonance_A2
results_attr2["best_resonance"] = best_resonance

print("Wyniki względem dwóch atraktorów:")
print(results_attr2.sort_values("best_resonance", ascending=False))

Wyniki względem dwóch atraktorów:
    novelty  feasibility  cost_efficiency  stability  label  distance_A1  \
0      0.90         0.85             0.80       0.75      1     0.050000   
4      0.75         0.70             0.65       0.60      1     0.327872   
6      0.92         0.90             0.85       0.88      1     0.108628   
1      0.88         0.80             0.78       0.70      1     0.115326   
10     0.83         0.79             0.81       0.72      1     0.122474   
8      0.78         0.82             0.74       0.76      1     0.143178   
5      0.35         0.40             0.45       0.30      0     0.936750   
9      0.28         0.35             0.38       0.32      0     1.020392   
2      0.20         0.30             0.40       0.35      0     1.074709   
3      0.25         0.20             0.30       0.25      0     1.182159   
11     0.18         0.22             0.33       0.27      0     1.190420   
7      0.15         0.25             0.20       0.22  

In [4]:
# ==========================================
# SEKCJA 4
# Hybryda: sieć + cechy atraktorowe
# ==========================================

# -------------------------------------------------
# 1. Budujemy nowe cechy atraktorowe
# -------------------------------------------------

distance_A1 = np.linalg.norm(X - attractor_1, axis=1).reshape(-1, 1)
distance_A2 = np.linalg.norm(X - attractor_2, axis=1).reshape(-1, 1)

resonance_A1 = (1 / (1 + distance_A1))
resonance_A2 = (1 / (1 + distance_A2))

# -------------------------------------------------
# 2. Łączymy surowe cechy z cechami atraktorowymi
# Oryginalnie mieliśmy 4 cechy
# Teraz dodajemy 4 kolejne:
# distance_A1, distance_A2, resonance_A1, resonance_A2
# -------------------------------------------------

X_hybrid = np.hstack([
    X,
    distance_A1,
    distance_A2,
    resonance_A1,
    resonance_A2
]).astype(np.float32)

hybrid_feature_names = [
    "novelty", "feasibility", "cost_efficiency", "stability",
    "distance_A1", "distance_A2", "resonance_A1", "resonance_A2"
]

df_hybrid = pd.DataFrame(X_hybrid, columns=hybrid_feature_names)
df_hybrid["label"] = y.astype(int)

print("Dane hybrydowe:")
print(df_hybrid)

# -------------------------------------------------
# 3. Budujemy nową sieć dla 8 cech wejściowych
# -------------------------------------------------

model_hybrid = models.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(12, activation="relu"),
    layers.Dense(6, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model_hybrid.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# -------------------------------------------------
# 4. Uczymy model hybrydowy
# -------------------------------------------------

history_hybrid = model_hybrid.fit(
    X_hybrid, y,
    epochs=120,
    verbose=0
)

# -------------------------------------------------
# 5. Predykcje modelu hybrydowego
# -------------------------------------------------

hybrid_pred = model_hybrid.predict(X_hybrid, verbose=0).flatten()
hybrid_class = (hybrid_pred > 0.5).astype(int)

results_final = df.copy()
results_final["base_nn_score"] = base_pred
results_final["base_nn_class"] = base_class
results_final["hybrid_nn_score"] = hybrid_pred
results_final["hybrid_nn_class"] = hybrid_class
results_final["distance_A1"] = distance_A1.flatten()
results_final["distance_A2"] = distance_A2.flatten()
results_final["resonance_A1"] = resonance_A1.flatten()
results_final["resonance_A2"] = resonance_A2.flatten()

print("\nPorównanie modelu bazowego i hybrydowego:")
print(results_final)

Dane hybrydowe:
    novelty  feasibility  cost_efficiency  stability  distance_A1  \
0      0.90         0.85             0.80       0.75     0.050000   
1      0.88         0.80             0.78       0.70     0.115326   
2      0.20         0.30             0.40       0.35     1.074709   
3      0.25         0.20             0.30       0.25     1.182159   
4      0.75         0.70             0.65       0.60     0.327872   
5      0.35         0.40             0.45       0.30     0.936750   
6      0.92         0.90             0.85       0.88     0.108628   
7      0.15         0.25             0.20       0.22     1.272360   
8      0.78         0.82             0.74       0.76     0.143178   
9      0.28         0.35             0.38       0.32     1.020392   
10     0.83         0.79             0.81       0.72     0.122474   
11     0.18         0.22             0.33       0.27     1.190420   

    distance_A2  resonance_A1  resonance_A2  label  
0      0.322025      0.952381    